# Notebook 03 — Covarianzas alternativas para el componente espacial

Este notebook resume la comparación entre cuatro funciones de covarianza para el componente espacial de GPBoost: exponencial, Matérn con $\nu=1.5$, Matérn con $\nu=2.5$ y gaussiana.

La comparación parte de una hipótesis metodológica simple: cuando el proceso simulado contiene discontinuidades o cambios abruptos, las covarianzas menos suaves deberían ajustarse mejor que las covarianzas demasiado lisas.

## 0. Configuración del entorno

La siguiente celda prepara el notebook para dos escenarios de uso:

- En Google Colab, clona el repositorio si todavía no está disponible.
- En ejecución local, asume que el notebook se corre desde la raíz del repositorio o desde la carpeta `notebooks/`.

Este notebook no reentrena modelos. Solo carga resultados ya calculados y genera una figura comparativa.

In [ ]:
from pathlib import Path
import subprocess
import sys


IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/linaq11/examen-2-gpboost.git"

if IN_COLAB:
    repo_dir = Path("/content/repo")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "-q", REPO_URL, str(repo_dir)],
            check=True,
        )
    %cd /content/repo
else:
    current_dir = Path.cwd()
    if current_dir.name == "notebooks":
        %cd ..

print(f"Directorio de trabajo: {Path.cwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


RANDOM_SEED = 1
DATA_DIR = Path("data")
MODEL_DIR = Path("modelos")
FIGURE_DIR = Path("figuras")

np.random.seed(RANDOM_SEED)

for directory in [DATA_DIR, MODEL_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

## 1. Carga de datos

Se cargan dos insumos:

1. `simulado_train_seed1.csv`, usado para conservar trazabilidad con el conjunto de entrenamiento de la simulación.
2. `tabla_covarianzas.csv`, que contiene los resultados agregados de RMSE para cada función de covarianza.

El análisis de este notebook se concentra en la segunda tabla.

In [ ]:
training_path = DATA_DIR / "simulado_train_seed1.csv"
covariance_table_path = DATA_DIR / "tabla_covarianzas.csv"

required_files = [training_path, covariance_table_path]
missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "No se encontraron los siguientes archivos requeridos: "
        + ", ".join(missing_files)
    )

training_data = pd.read_csv(training_path)
covariance_results = pd.read_csv(covariance_table_path)

feature_columns = ["X1", "X2"]
coord_columns = ["s_1", "s_2"]

X = training_data[feature_columns].to_numpy()
y = training_data["y"].to_numpy()
coords = training_data[coord_columns].to_numpy()

covariance_results

## 2. Comparación de resultados

La figura resume el RMSE obtenido sobre el grid de prueba de 50 × 50. Las funciones se presentan en orden aproximado de suavidad creciente: exponencial, Matérn $\nu=1.5$, Matérn $\nu=2.5$ y gaussiana.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

bar_positions = np.arange(len(covariance_results))
bars = ax.bar(
    bar_positions,
    covariance_results["RMSE"],
)

for position, rmse_value in zip(bar_positions, covariance_results["RMSE"]):
    ax.text(
        position,
        rmse_value + 0.005,
        f"{rmse_value:.4f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax.set_xticks(bar_positions)
ax.set_xticklabels(covariance_results["cov_function"], rotation=15, ha="right")
ax.set_ylabel("RMSE en grid de prueba 50 × 50")
ax.set_title("Comparación de funciones de covarianza")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()

output_path = FIGURE_DIR / "fig6_covarianzas_RMSE.png"
fig.savefig(output_path, dpi=150, bbox_inches="tight")

plt.show()

print(f"Figura guardada en: {output_path}")

## 3. Interpretación

El RMSE aumenta de manera ordenada a medida que la función de covarianza impone mayor suavidad. La covarianza exponencial, equivalente a una Matérn con $\nu=0.5$, obtiene el menor error. Después aparecen Matérn $\nu=1.5$, Matérn $\nu=2.5$ y, finalmente, la gaussiana.

Este patrón es coherente con la simulación usada en el ejercicio, ya que la función generadora contiene una discontinuidad en $x_1=0.5$. En ese contexto, una covarianza demasiado suave puede sobrerregular el componente espacial y perder capacidad para representar cambios abruptos.

La lectura metodológica es directa: cuando los datos presentan rugosidad o discontinuidades, conviene evaluar covarianzas menos suaves antes de asumir una estructura gaussiana. Para este caso, la elección exponencial usada como referencia resulta consistente con la forma del proceso simulado.

## 4. Nota de sintaxis para GPBoost

En GPBoost, la familia Matérn se define con `cov_function="matern"`. El parámetro de suavidad $\nu$ se pasa mediante `cov_fct_shape`, no como parte del nombre de la función de covarianza.

In [ ]:
# Ejemplos de sintaxis para documentar la configuración de GPBoost.
# Estas líneas no se ejecutan porque el notebook solo resume resultados ya calculados.

# import gpboost as gpb
#
# gp_matern_15 = gpb.GPModel(
#     gp_coords=coords,
#     cov_function="matern",
#     cov_fct_shape=1.5,
# )
#
# gp_matern_25 = gpb.GPModel(
#     gp_coords=coords,
#     cov_function="matern",
#     cov_fct_shape=2.5,
# )